## Install All the Required Packages

In [1]:
!pip -q install langchain langchain_community langchain-openai langchain-pinecone langchain-text-splitters pinecone-client==2.2.4 pypdf openai tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.4/179.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/2

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain_openai import OpenAI
from langchain_pinecone import PineconeVectorStore
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import os
import sys

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [ ]:
!mkdir pdfs

In [ ]:
loader = PyPDFDirectoryLoader("pdfs")
data = loader.load()

In [ ]:
data

### Split the Extracted Data into Text Chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chuck_size = 500, chuck_overlap = 20)

In [ ]:
text_chunks = text_splitter.split_documents(data)

In [ ]:
text_chunks

Download the Embeddings

In [ ]:
os.environ['OPENAI_API_KEY'] = "Your_API_Key"

In [ ]:
embeddings = OpenAIEmbeddings()

In [ ]:
result = embeddings.embed_query("Hello")

Initialize Pinecone

In [ ]:
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY', 'Your_Key')
PINECONE_API_ENV = os.environ.get('PINECODE_API_ENV', 'gcp-starter')

In [ ]:
#initialize pinecone
pinecone.init(
    api_key = PINECONE_API_KEY,
    environment = PINECONE_API_ENV
)
index_name = "test" # put the name of your pinecone index here

Create Embeddings for each of the Text Chunk

In [ ]:
docsearch = Pinecone.from_texts([t.page_content for t in text_chunks], embeddings, index_name = index_name)

Similarity Search

In [ ]:
query = "YOLOv7 outperforms which models"

In [ ]:
docs = docsearch.similarity_search(query, k = 3)

## Creating a LLM Model Wrapper

In [ ]:
llm = OpenAI()

In [ ]:
qa = RetrievalQA.from_chain_type(llm = llm, chain_type = "stuff", retriever = docsearch.as_retriever())

## QA

In [ ]:
query = "YOLOv7 outperforms which models"

In [ ]:
qa.run(query)

In [ ]:
query = "Rachel Green Experience"

In [ ]:
qa.run(query)

In [ ]:
while True:
  user_input = input(f"What is your Question? ")
  if user_input = 'exit':
    print("Exiting...")
    sys.exit()
  if user_input == '':
    continue
  result = qa({'query': user_input})
  print(f"Answer: {result['result']}")